# 🧾 Matriculator · Demo de lenguajes de marcado y formatos (Paso 4)

**Responsable:** ✏️ *Nombre del compañero/a*

Contenido:
1. Librerías principales de Python para leer **HTML, CSV y JSON**
2. Tabla con las librerías equivalentes en **R**
3. HTML inventado de la interfaz de Matriculator → parseo
4. JSON inventado con datos de entrada sintéticos → parseo

> Todos los datos son **ficticios**.

## 1 · Librerías de Python para leer HTML, CSV y JSON

| Formato | Librería | Tipo | Uso típico |
|---|---|---|---|
| HTML | `html.parser` | Estándar | Parser básico incluido en Python |
| HTML | `BeautifulSoup` (bs4) | Externa | Navegar y extraer etiquetas de forma sencilla |
| HTML | `lxml` | Externa | Parser muy rápido (HTML y XML) |
| CSV | `csv` | Estándar | Leer/escribir filas |
| CSV | `pandas.read_csv` | Externa | Cargar tablas en un DataFrame |
| JSON | `json` | Estándar | `json.load` / `json.loads` |
| JSON | `pandas.read_json` / `json_normalize` | Externa | JSON tabular o anidado a DataFrame |

✏️ *TODO: revisar y citar la documentación oficial de cada librería en el README.*

In [1]:
import json, csv, io
import pandas as pd
from bs4 import BeautifulSoup

## 2 · Equivalentes en R

| Formato | Python | R |
|---|---|---|
| HTML | `BeautifulSoup`, `lxml` | `rvest`, `xml2` |
| CSV | `csv`, `pandas.read_csv` | `read.csv()` (base), `readr::read_csv()`, `data.table::fread()` |
| JSON | `json`, `pandas.read_json` | `jsonlite::fromJSON()`, `rjson` |

✏️ *TODO: comprobar en CRAN y añadir fuentes.*

## 3 · HTML inventado de la interfaz de Matriculator

In [2]:
html_interfaz = """
<!DOCTYPE html>
<html lang="es">
<head><title>Matriculator · Panel de pruebas</title></head>
<body>
  <h1>Matriculator</h1>
  <form id="subida" action="/api/leer" method="post">
    <label for="img">Imagen de prueba</label>
    <input type="file" id="img" name="imagen" accept=".jpg,.png">
    <label for="camara">Cámara</label>
    <select id="camara" name="camara_id">
      <option value="CAM-01">Entrada norte</option>
      <option value="CAM-02">Salida sur</option>
    </select>
    <button type="submit">Analizar</button>
  </form>
  <table id="resultados">
    <tr><th>Matrícula</th><th>Confianza</th><th>Estado</th></tr>
    <tr><td>0000XXX</td><td>0.93</td><td>OK</td></tr>
    <tr><td>-</td><td>0.41</td><td>REVISION_HUMANA</td></tr>
  </table>
</body>
</html>
"""

soup = BeautifulSoup(html_interfaz, "html.parser")
print("Título:", soup.title.string)
print("Formulario envía a:", soup.find("form")["action"])
print("Cámaras:", [o["value"] for o in soup.select("#camara option")])

# La tabla HTML también puede pasar a DataFrame
filas = [[td.get_text() for td in tr.find_all(["th", "td"])] for tr in soup.select("#resultados tr")]
pd.DataFrame(filas[1:], columns=filas[0])

Título: Matriculator · Panel de pruebas
Formulario envía a: /api/leer
Cámaras: ['CAM-01', 'CAM-02']


,Matrícula,Confianza,Estado
0,0000XXX,0.93,OK
1,-,0.41,REVISION_HUMANA


## 4 · JSON inventado con datos de entrada sintéticos

In [3]:
json_entrada = """
{
  "lote": "PRUEBA-2026-10",
  "origen": "banco_de_pruebas_sintetico",
  "imagenes": [
    {"id": 1, "ruta_imagen": "pruebas/imagen_001.jpg", "camara": {"id": "CAM-01", "resolucion": "1920x1080", "iluminacion": "dia"}},
    {"id": 2, "ruta_imagen": "pruebas/imagen_002.jpg", "camara": {"id": "CAM-01", "resolucion": "1920x1080", "iluminacion": "noche"}},
    {"id": 3, "ruta_imagen": "pruebas/imagen_003.png", "camara": {"id": "CAM-02", "resolucion": "1280x720", "iluminacion": "dia"}}
  ]
}
"""

datos = json.loads(json_entrada)
print("Lote:", datos["lote"], "| nº imágenes:", len(datos["imagenes"]))
pd.json_normalize(datos["imagenes"])   # aplana el objeto anidado 'camara' 

Lote: PRUEBA-2026-10 | nº imágenes: 3


,id,ruta_imagen,camara.id,camara.resolucion,camara.iluminacion
0,1,pruebas/imagen_001.jpg,CAM-01,1920x1080,dia
1,2,pruebas/imagen_002.jpg,CAM-01,1920x1080,noche
2,3,pruebas/imagen_003.png,CAM-02,1280x720,dia


## 5 · (Extra) CSV de registros

In [4]:
csv_registros = """fecha,camara_id,matricula,confianza,estado
2026-10-01T09:15,CAM-01,0000XXX,0.93,OK
2026-10-01T09:16,CAM-01,,0.41,REVISION_HUMANA
"""
df = pd.read_csv(io.StringIO(csv_registros))
df

,fecha,camara_id,matricula,confianza,estado
0,2026-10-01T09:15,CAM-01,0000XXX,0.93,OK
1,2026-10-01T09:16,CAM-01,NaN,0.41,REVISION_HUMANA


## 6 · Conclusión

✏️ *TODO: explicar en qué parte de Matriculator interviene cada formato (HTML → interfaz, JSON → API, CSV → registros/Trends, XML → anotaciones, Markdown → documentación).*